## WP014 — Confirming lineup continuity on fresh matches

See `README.md`. **Heavy compute — run this yourself.** 34 new walk-forward windows, each testing 3 rounds that no WP001 window ever tested (1,086 new held-out matches), four arms, 136 fits. The primary test was declared before any of it ran:

**Primary:** `continuity − baseline`, paired RPS on the new matches only, 95% bootstrap CI entirely below zero. **Secondary:** `continuity_lineup_loose_combo − lineup_loose_combo`, same rule. Everything else is exploratory.

In [1]:
import pickle, sys, time
from pathlib import Path

import numpy as np
import pandas as pd

from football_model.evaluation import market as mk
from football_model.evaluation.windows import tested_rounds
from football_model.types.model_data import ModelConfig

REPO = Path('/Users/hadiahmed/Documents/projects/football-predictor')
WP003 = REPO / 'work_products' / 'wp003_bookmaker_benchmark'
WP013 = REPO / 'work_products' / 'wp013_lineup_continuity'
WP014 = REPO / 'work_products' / 'wp014_continuity_confirmation'
SCRIPT = REPO / 'scripts' / 'run_cv_window.py'
sys.path.insert(0, str(REPO / 'scripts'))
from run_cv_window import run_windows_concurrent  # noqa: E402

DATA_PATH = WP014 / 'cv_shared_data.pkl'
with open(DATA_PATH, 'rb') as f:
    shared = pickle.load(f)
df_cv, windows = shared['df_cv'], shared['windows']
old_windows = pickle.load(open(WP013 / 'cv_shared_data.pkl', 'rb'))['windows']
assert 'continuity_table' in shared and 'lineup_dev_table' in shared

# The whole point of this WP: none of these matches was ever held out before.
new_rounds, old_rounds = tested_rounds(windows), tested_rounds(old_windows)
assert new_rounds.isdisjoint(old_rounds), 'new windows re-use rounds the original windows tested'
home = df_cv[df_cv['is_home'] == 1]
print(f'{len(windows)} new windows; {int(home["round"].isin(new_rounds).sum())} new held-out matches '
      f'(original: {int(home["round"].isin(old_rounds).sum())}); rounds disjoint: {new_rounds.isdisjoint(old_rounds)}')

odds = pd.read_pickle(WP003 / 'odds_raw.pkl').dropna(subset=['Date', 'FTR']).reset_index(drop=True)

LOOSE = dict(init_scale=0.30, home_adv_sd=0.06, sigma_att=0.020, sigma_def=0.020)   # WP005's loose_combo
ARMS = {
    'baseline':                      {},
    'lineup_loose_combo':            {'use_lineup_xg': True, **LOOSE},
    'continuity':                    {'use_continuity': True},
    'continuity_lineup_loose_combo': {'use_continuity': True, 'use_lineup_xg': True, **LOOSE},
}
BASE = dict(clip_theta=5.0, center_team_strength=False, use_dixon_coles=True, use_xG=True)
for name, ov in ARMS.items():
    ModelConfig(**BASE, **ov)
print('arms:', list(ARMS))

MAX_WORKERS = 3     # 12 chain threads on a 12-core laptop throttles; see WP012/WP013 discussion
WINDOW_TIMEOUT = 1800
ALL_WINDOWS = list(range(1, len(windows) + 1))

def load_ckpt(p):
    p = Path(p)
    return pickle.load(open(p, 'rb')) if p.exists() else {'results': [], 'cv_match_predictions': []}

34 new windows; 1086 new held-out matches (original: 401); rounds disjoint: True
arms: ['baseline', 'lineup_loose_combo', 'continuity', 'continuity_lineup_loose_combo']


### Time one window first

Each window tests 3 rounds instead of WP001's 1, but that only affects prediction (cheap), not sampling. The one-window figure should be close to WP013's.

In [2]:
t0 = time.time()
run_windows_concurrent(SCRIPT, DATA_PATH, WP014 / 'cv_checkpoint_continuity.pkl', [ALL_WINDOWS[0]],
                       config_overrides=ARMS['continuity'], max_workers=1, timeout=WINDOW_TIMEOUT)
print(f'one window: {time.time() - t0:.1f}s')

[window 1/34] training rounds 1-38 (use_xg=True, use_dc=True, overrides={'use_continuity': True})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 1:   5%|▌         | 200/4000 [00:07<01:39, 38.36it/s]

Running chain 0:  15%|█▌        | 600/4000 [00:08<00:32, 105.59it/s][A

Running chain 0:  20%|██        | 800/4000 [00:09<00:24, 132.17it/s][A

Running chain 1:  20%|██        | 800/4000 [00:10<00:25, 125.31it/s]

Running chain 0:  30%|███       | 1200/4000 [00:11<00:16, 166.93it/s]

Running chain 0:  35%|███▌      | 1400/4000 [00:12<00:14, 180.60it/s]

Running chain 0:  40%|████      | 1600/4000 [00:13<00:12, 193.10it/s]

Running chain 0:  45%|████▌     | 1800/4000 [00:14<00:10, 205.04it/s]

Running chain 0:  50%|█████     | 2000/4000 [00:15<00:09, 205.41it/s]

Running chain 0:  55%|█████▌    | 2200/4000 [00:16<00:08, 214.34it/s]

Running chain 0:  60%|██████    | 2400/4000 [00:16<00:07, 219.04it/s]

Running chain 0:  65%|██████▌   | 2600/4000 [00:17<00:06, 224.7

[window 1] MAE=0.819 LL_improvement=3.32
[window 1] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp014_continuity_confirmation/.tmp_cv_checkpoint_continuity_w1.pkl
  [window 1] done and merged
one window: 38.2s


### Run all four arms on all 34 windows

The checkpoints resume: windows already present are skipped, so this can be stopped and re-run in batches.

In [3]:
t0 = time.time()
for name, ov in ARMS.items():
    run_windows_concurrent(SCRIPT, DATA_PATH, WP014 / f'cv_checkpoint_{name}.pkl', ALL_WINDOWS,
                           config_overrides=ov, max_workers=MAX_WORKERS, timeout=WINDOW_TIMEOUT)
print(f'\nwall time: {(time.time() - t0) / 60:.1f} min')
for name in ARMS:
    print(f'  {name}: {len(load_ckpt(WP014 / f"cv_checkpoint_{name}.pkl")["results"])}/{len(ALL_WINDOWS)}')

[window 2/34] training rounds 1-43 (use_xg=True, use_dc=True, overrides={})
[window 1/34] training rounds 1-38 (use_xg=True, use_dc=True, overrides={})
[window 3/34] training rounds 1-48 (use_xg=True, use_dc=True, overrides={})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:02<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:02<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:02<?, ?it/s]

Running chain 0:   5%|▌         | 200/4000 [00:09<02:19, 27.30it/s]

Running chain 2:   5%|▌         | 200/4000 [00:09<02:20, 26.97it/s]

Running chain 0:   5%|▌         | 200/4000 [00:11<02:47, 22.66it/s]

Running chain 2:  10%|█         | 400/4000 [00:11<01:09, 51.92it/s]

Running chain 2:   5%|▌         | 200/4000 [00:11<02:49, 22.44it/s]

Running chain 1:  10%|█         | 400/4000 [00:12<01:19, 45.29it/s]

Running chain 0:  10%|█         | 400/4000 [00:12<01:21, 44.43it/s]

Running chain 1:  20%|██        | 800/4000 [00

[window 1] MAE=0.822 LL_improvement=3.22
[window 1] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp014_continuity_confirmation/.tmp_cv_checkpoint_baseline_w1.pkl


Running chain 0: 100%|██████████| 4000/4000 [00:41<00:00, 97.23it/s] 


  [window 1] done and merged
[window 4/34] training rounds 1-53 (use_xg=True, use_dc=True, overrides={})


...

[window 32] MAE=0.789 LL_improvement=5.45
[window 32] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp014_continuity_confirmation/.tmp_cv_checkpoint_continuity_lineup_loose_combo_w32.pkl
  [window 32] done and merged


Running chain 1:   5%|▌         | 200/4000 [01:04<19:47,  3.20it/s]

Running chain 0:  65%|██████▌   | 2600/4000 [03:14<01:15, 18.53it/s]

Running chain 0:  15%|█▌        | 600/4000 [01:15<05:38, 10.04it/s]]

Running chain 0:  70%|███████   | 2800/4000 [03:24<01:03, 19.00it/s][A

Running chain 2:  10%|█         | 400/4000 [01:20<08:48,  6.81it/s]

Running chain 1:  70%|███████   | 2800/4000 [03:28<01:03, 18.98it/s]

Running chain 0:  75%|███████▌  | 3000/4000 [03:34<00:52, 19.13it/s][A

Running chain 1:  75%|███████▌  | 3000/4000 [03:39<00:52, 19.02it/s]

Running chain 1:  80%|████████  | 3200/4000 [03:49<00:41, 19.17it/s]

Running chain 2:  25%|██▌       | 1000/4000 [01:42<03:43, 13.43it/s]

Running chain 1:  85%|████████▌ | 3400/4000 [03:59<00:30, 19.39it/s]

Running chain 2:  30%|███       | 1200/4000 [01:52<03:07, 14.91it/s]

Running chain 1:  90%|█████████ | 3600/4000 [04:09<00:20, 19.50it/s]

Running chain 2:  35%|███▌      | 1400/4000 [02:05<02:48, 15.40it/s]

Running chain 1:  

[window 33] MAE=0.979 LL_improvement=-1.90
[window 33] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp014_continuity_confirmation/.tmp_cv_checkpoint_continuity_lineup_loose_combo_w33.pkl
  [window 33] done and merged



Running chain 1:  55%|█████▌    | 2200/4000 [02:51<01:35, 18.79it/s]

Running chain 1:  60%|██████    | 2400/4000 [03:00<01:22, 19.34it/s]

Running chain 1:  65%|██████▌   | 2600/4000 [03:10<01:10, 19.92it/s]

Running chain 1:  70%|███████   | 2800/4000 [03:19<00:59, 20.25it/s]

Running chain 1:  75%|███████▌  | 3000/4000 [03:28<00:48, 20.56it/s]

Running chain 1:  80%|████████  | 3200/4000 [03:38<00:38, 20.61it/s]

Running chain 1:  85%|████████▌ | 3400/4000 [03:48<00:28, 20.70it/s]

Running chain 1:  90%|█████████ | 3600/4000 [03:57<00:19, 20.79it/s]

Running chain 0: 100%|██████████| 4000/4000 [04:03<00:00, 16.45it/s]

Running chain 1:  95%|█████████▌| 3800/4000 [04:06<00:09, 21.44it/s]

Running chain 2: 100%|██████████| 4000/4000 [04:09<00:00, 16.06it/s]

Running chain 1: 100%|██████████| 4000/4000 [04:13<00:00, 15.77it/s]
We recommend running at least 4 chains for robust computation of convergence diagnostics


[window 34] MAE=0.921 LL_improvement=2.88
[window 34] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp014_continuity_confirmation/.tmp_cv_checkpoint_continuity_lineup_loose_combo_w34.pkl
  [window 34] done and merged

wall time: 135.5 min
  baseline: 34/34
  lineup_loose_combo: 34/34
  continuity: 34/34
  continuity_lineup_loose_combo: 34/34


## Analysis — new matches only

Every arm's held-out predictions, aligned to the fixtures and checked against `df_cv`'s own goals (the alignment assertion raises if any match is misplaced). RPS per match from the model's win/draw/loss probabilities and the actual result; no odds needed for the arm comparison, so all new matches count. Pinnacle's gap is reported on the subset with closing odds, for context.

In [7]:
def fixtures_checked(ckpt, wins):
    fx = mk.model_fixtures(df_cv, wins, ckpt)
    ds = df_cv.sort_values('datetime').reset_index(drop=True)
    rounds = tested_rounds(wins)
    exp = ds[(ds['is_home'] == 1) & (ds['round'].isin(rounds))]
    assert len(exp) == len(fx), (len(exp), len(fx))
    assert np.array_equal(fx['goals_home'].to_numpy(), exp['goals'].to_numpy()), 'fixtures misaligned (home goals)'
    assert np.array_equal(fx['goals_away'].to_numpy(), exp['goals_against'].to_numpy()), 'fixtures misaligned (away goals)'
    return fx

def result_onehot(fx):
    gh, ga = fx['goals_home'].to_numpy(), fx['goals_away'].to_numpy()
    return mk.outcome_onehot(np.where(gh > ga, 'H', np.where(gh == ga, 'D', 'A')))

def arm_rps(fx):
    return mk.rps(fx[['p_home_model', 'p_draw_model', 'p_away_model']].to_numpy(), result_onehot(fx))

def analyse(prefix, wins, label):
    ckpts = {n: load_ckpt(WP014 / f'cv_checkpoint_{prefix}{n}.pkl') for n in ARMS}
    missing = [n for n, c in ckpts.items() if len(c['results']) < len(wins)]
    if missing:
        print('not complete yet:', {n: len(ckpts[n]['results']) for n in missing}); return None
    FX = {n: fixtures_checked(c, wins) for n, c in ckpts.items()}
    key = ['date', 'home_fd', 'away_fd']
    for n, fx in FX.items():
        assert fx[key].equals(FX['baseline'][key]), f'{n} covers different matches'
    R = {n: arm_rps(fx) for n, fx in FX.items()}
    n_m = len(R['baseline'])
    print(f'{label}: {n_m} held-out matches')
    rows = []
    for n in ARMS:
        rows.append({'arm': n, 'rps': R[n].mean()})
    print(pd.DataFrame(rows).round(4).to_string(index=False))
    print()
    out = []
    for lab, a, b in [('PRIMARY   continuity − baseline', 'continuity', 'baseline'),
                      ('secondary continuity_lineup_loose_combo − lineup_loose_combo', 'continuity_lineup_loose_combo', 'lineup_loose_combo'),
                      ('context   lineup_loose_combo − baseline', 'lineup_loose_combo', 'baseline')]:
        d = R[a] - R[b]
        m, lo, hi = mk.bootstrap_ci(d, 5000)
        out.append({'paired RPS difference (negative = first is better)': lab, 'mean': m, 'lo': lo, 'hi': hi,
                    'better_on': (d < 0).mean(), 'verdict': ('HIT' if hi < 0 else 'no hit') if lab.startswith(('PRIMARY', 'secondary')) else ''})
    res = pd.DataFrame(out)
    print(res.round(5).to_string(index=False))
    se = (res.loc[0, 'hi'] - res.loc[0, 'lo']) / (2 * 1.96)
    print(f'\nprimary standard error {se:.5f}; smallest effect detected 80% of the time about {2.8 * se:.5f}')
    return FX, R, ckpts

new_out = analyse('', windows, 'NEW matches (primary analysis)')

NEW matches (primary analysis): 1086 held-out matches
                          arm    rps
                     baseline 0.2059
           lineup_loose_combo 0.2051
                   continuity 0.2056
continuity_lineup_loose_combo 0.2049

          paired RPS difference (negative = first is better)     mean       lo      hi  better_on verdict
                             PRIMARY   continuity − baseline -0.00025 -0.00097 0.00046    0.49079  no hit
secondary continuity_lineup_loose_combo − lineup_loose_combo -0.00026 -0.00095 0.00042    0.49355  no hit
                     context   lineup_loose_combo − baseline -0.00073 -0.00151 0.00006    0.55341        

primary standard error 0.00037; smallest effect detected 80% of the time about 0.00102


### Exploratory: beta_continuity, calendar years, and the Pinnacle gap

Not part of the pre-declared tests. `beta_continuity` per window (the hypothesis says negative); the primary difference by calendar year; and each arm's gap to Pinnacle's close on the new matches that have closing odds.

In [8]:
if new_out:
    FX, R, ckpts = new_out
    for n in ('continuity', 'continuity_lineup_loose_combo'):
        b = np.array([r['beta_continuity'] for r in ckpts[n]['results']], float)
        print(f'beta_continuity, {n}: mean {b.mean():+.4f}  range [{b.min():+.4f}, {b.max():+.4f}]  negative in {(b < 0).sum()}/{len(b)} windows')
    d = R['continuity'] - R['baseline']
    yr = pd.to_datetime(FX['baseline']['date']).dt.year
    print(pd.DataFrame({'year': yr, 'diff': d}).groupby('year')['diff'].agg(['count', 'mean']).round(5).T.to_string())
    f, s = mk.half_masks(FX['baseline']['date'])
    print(f'halves: first {d[f].mean():+.5f}  second {d[s].mean():+.5f}')

    need = [f'PSC{o}' for o in mk.OUTCOMES]
    Jb = {n: mk.join_odds(fx, odds, required_cols=need) for n, fx in FX.items()}
    ymat = mk.outcome_onehot(Jb['baseline']['FTR'])
    pin = mk.devig(mk.odds_matrix(Jb['baseline'], 'PSC'))
    rp = mk.rps(pin, ymat)
    print(f'\nPinnacle-covered subset: {len(rp)} matches, Pinnacle RPS {rp.mean():.4f}')
    for n, j in Jb.items():
        r = mk.rps(j[['p_home_model', 'p_draw_model', 'p_away_model']].to_numpy(), ymat)
        m, lo, hi = mk.bootstrap_ci(r - rp, 5000)
        print(f'  {n:<30} RPS {r.mean():.4f}  gap {m:+.4f} [{lo:+.4f}, {hi:+.4f}]')

beta_continuity, continuity: mean -0.0340  range [-0.0540, +0.0253]  negative in 31/34 windows
beta_continuity, continuity_lineup_loose_combo: mean -0.0328  range [-0.0526, +0.0255]  negative in 31/34 windows
year       2021       2022       2023       2024       2025       2026
count  84.00000  193.00000  251.00000  220.00000  226.00000  112.00000
mean    0.00152   -0.00019   -0.00157   -0.00126    0.00017    0.00241
halves: first -0.00061  second +0.00010

Pinnacle-covered subset: 998 matches, Pinnacle RPS 0.1982
  baseline                       RPS 0.2056  gap +0.0074 [+0.0035, +0.0113]
  lineup_loose_combo             RPS 0.2048  gap +0.0067 [+0.0029, +0.0104]
  continuity                     RPS 0.2052  gap +0.0070 [+0.0031, +0.0108]
  continuity_lineup_loose_combo  RPS 0.2044  gap +0.0063 [+0.0025, +0.0100]


### Exploratory: pooled old + new (NOT independent of the original selection)

The original 401 matches are where the idea came from, so a pooled estimate is optimistic by construction. Shown only to see how much the new matches move the WP013 number.

In [9]:
if new_out:
    old_ck = {n: pickle.load(open(WP013 / f'cv_checkpoint_full_{n}.pkl', 'rb')) for n in ARMS}
    R_old = {n: arm_rps(fixtures_checked(c, old_windows)) for n, c in old_ck.items()}
    _, R_new, _ = new_out
    for lab, a, b in [('continuity − baseline', 'continuity', 'baseline'),
                      ('continuity_lineup_loose_combo − lineup_loose_combo', 'continuity_lineup_loose_combo', 'lineup_loose_combo')]:
        rows = []
        for part, Rx in [('original 401', R_old), ('new', R_new), ('pooled', {n: np.concatenate([R_old[n], R_new[n]]) for n in ARMS})]:
            d = Rx[a] - Rx[b]
            m, lo, hi = mk.bootstrap_ci(d, 5000)
            rows.append({'matches': part, 'n': len(d), 'mean': m, 'lo': lo, 'hi': hi})
        print(lab); print(pd.DataFrame(rows).round(5).to_string(index=False)); print()

continuity − baseline
     matches    n     mean       lo       hi
original 401  401 -0.00116 -0.00228 -0.00003
         new 1086 -0.00025 -0.00097  0.00046
      pooled 1487 -0.00050 -0.00110  0.00012

continuity_lineup_loose_combo − lineup_loose_combo
     matches    n     mean       lo      hi
original 401  401 -0.00107 -0.00215 0.00000
         new 1086 -0.00026 -0.00095 0.00042
      pooled 1487 -0.00048 -0.00106 0.00011

